In [3]:
import xarray as xr
import pandas as pd
import sqlalchemy
import pygrib
import sqlite3

import numpy as np
from pathlib import Path

import numpy as np



1. Get ERA5 data with era5Get.py
2. Starts with 2t,2d,pcp on a .10 degree grid (2981 points total)
3. data is 3 hourly 
4. grib2db.ipynb -> /home/joe/work/Fire/ML/New/DB/era5_daily_2982_{TARGET_VAR}.sqlite
6. 
8. 
9. Monthly means are then computed by variable  -> /home/joe/work/Fire/ML/Data/DB/era5_means_2982_{TARGET_VAR}.sqlite
10. Monthly means combined into 1 table, and RH and VPD are computed

## Compute Monthly Means

### Compute Daily Means 

In [7]:
import sqlite3
from pathlib import Path
import pandas as pd

sqlite_path_2t = Path(f"/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_2t.sqlite")
sqlite_path_2d = Path(f"/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_2d.sqlite")
table_name = "daily_data"



sqlite_path_out_rh = Path(f"/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_RH.sqlite")
table_name = "daily_data"
latlons = pd.read_csv("latlons_42x71.csv")
sqlite_path_out_vpd = Path(f"/home/joe/work/Fire/ML/Data/DB/era5_daily_2982_VPD.sqlite")
table_name = "daily_data"


with sqlite3.connect(sqlite_path_out_rh) as conn:
    create_sql = """
    CREATE TABLE IF NOT EXISTS daily_data (
        datetime TEXT NOT NULL,
        point_id INTEGER NOT NULL,
        variable TEXT NOT NULL,
        '0' REAL,
        '3' REAL,
        '6' REAL,
        '9' REAL,
        '12' REAL,
        '15' REAL,
        '18' REAL,
        '21' REAL,
        PRIMARY KEY (datetime, point_id)
    );
    """
    conn.execute(create_sql)

with sqlite3.connect(sqlite_path_out_vpd) as conn:
    create_sql = """
        CREATE TABLE IF NOT EXISTS daily_data (
        datetime TEXT NOT NULL,
        point_id INTEGER NOT NULL,
        variable TEXT NOT NULL,
        '0' REAL,
        '3' REAL,
        '6' REAL,
        '9' REAL,
        '12' REAL,
        '15' REAL,
        '18' REAL,
        '21' REAL,
        PRIMARY KEY (datetime, point_id)
    );
    """
    conn.execute(create_sql)



In [ ]:
# if sqlite_path_out_rh.exists():
#     with sqlite3.connect(sqlite_path_out_rh) as conn:
#         max_yrmo = pd.read_sql_query(f"SELECT max(yrmo) FROM {table_name}", conn).values[0][0]
#         if max_yrmo is None:
#             max_yrmo = 0
#         else:
#             max_yrmo = int(max_yrmo)*100+31
#         print(f"Loaded max yrmo: {max_yrmo} from {sqlite_path_out_rh}")
# else:
#     max_yrmo = 0
#     print(f"No existing data found at {sqlite_path_out_rh}. Starting fresh.")  

Loaded max yrmo: 0 from /home/joe/work/Fire/ML/Data/DB/era5_daily_2982_RH.sqlite


In [8]:
import numpy as np

def rh_from_t_td(t, td, units="C"):
    """Compute relative humidity (%) from air temperature and dew point.

    Parameters
    ----------
    t : float
        Air temperature.
    td : float 
        Dew point temperature (same units as t).
    units : str
        "C" for Celsius (default) or "K" for Kelvin.
    """
    # t = np.asarray(t, dtype=float)
    # td = np.asarray(td, dtype=float)

    if units.upper() == "K":
        t = t - 273.15
        td = td - 273.15
    elif units.upper() != "C":
        raise ValueError("units must be 'C' or 'K'")

    # Magnus formula (valid for typical atmospheric conditions)
    rh = 100.0 * np.exp((17.625 * td) / (243.04 + td)) / np.exp((17.625 * t) / (243.04 + t))
    return round(rh, 1)

# Example
def vpd_from_t_td(t, td, units="C", out_units="kPa"):
    """Compute vapor pressure deficit from air temperature and dew point.

    VPD = es(T) - ea, where ea = es(Td).
    Inputs can be scalars, NumPy arrays, or pandas Series.
    """
    # t = np.asarray(t, dtype=float)
    # td = np.asarray(td, dtype=float)

    if units.upper() == "K":
        t = t - 273.15
        td = td - 273.15
    elif units.upper() != "C":
        raise ValueError("units must be 'C' or 'K'")

    es = 0.6108 * np.exp((17.27 * t) / (t + 237.3))
    ea = 0.6108 * np.exp((17.27 * td) / (td + 237.3))
    vpd_kpa = np.maximum(es - ea, 0.0)

    if out_units.lower() == "kpa":
        return vpd_kpa
    if out_units.lower() == "pa":
        return vpd_kpa * 1000.0
    raise ValueError("out_units must be 'kPa' or 'Pa'")

# Compute VPD columns for each 3-hour slot



In [36]:
conn2t =  sqlite3.connect(sqlite_path_2t) 
conn2d = sqlite3.connect(sqlite_path_2d) 

hours = [str(hr) for hr in range(0,22,3)]
print(hours)

with sqlite3.connect(sqlite_path_out_rh) as conn:
    npoints = len(latlons)
    for i, row in latlons.iterrows():
        lat = row['lat']
        lon = row['lon']
        pid = row['point']
        t = pd.read_sql_query(f"SELECT * FROM {table_name} where point_id = {pid} and datetime > {max_yrmo}", conn2t)
        td = pd.read_sql_query(f"SELECT * FROM {table_name} where point_id = {pid} and datetime > {max_yrmo}", conn2d)
    #  sample = pd.read_sql_query(f"SELECT * FROM {table_name} where valid_time < '1990-01-04-00'", conn)
        print(f"{i+1}/{npoints} Processing lat {lat}, lon {lon} with {len(t)} records")  
        df = pd.merge(t, td, on='datetime', suffixes=('_t', '_td'))
        cols = ['0','3','6','9','12','15','18','21']

        for col in cols:
            df[f"rh_{col}"] = rh_from_t_td(df[f"{col}_t"], df[f"{col}_td"], units="K")
            df[f"vpd_{col}"] = vpd_from_t_td(df[f"{col}_t"], df[f"{col}_td"], units="K", out_units="kPa")

        df_rh = df[['datetime', 'point_id_t', 'rh_0', 'rh_3', 'rh_6', 'rh_9', 'rh_12', 'rh_15', 'rh_18', 'rh_21']]
        df_rh['variable'] = 'rh'
        df_vpd = df[['datetime', 'point_id_t', 'vpd_0', 'vpd_3', 'vpd_6', 'vpd_9', 'vpd_12', 'vpd_15', 'vpd_18', 'vpd_21']]
        df_vpd['variable'] = 'vpd'

        for col in cols:
            df_rh.rename(columns={f"rh_{col}": f"{col}"}, inplace=True)
            df_vpd.rename(columns={f"vpd_{col}": f"{col}"}, inplace=True)
 
    
        with sqlite3.connect(sqlite_path_out_rh) as conn2:
            df_rh.to_sql(table_name, conn2, index=False, if_exists='append')
            
        with sqlite3.connect(sqlite_path_out_vpd) as conn2:
            df_vpd.to_sql(table_name, conn2, index=False, if_exists='append')

        break   

    

['0', '3', '6', '9', '12', '15', '18', '21']
1/2982 Processing lat 41.1, lon -109.0 with 12964 records


/tmp/ipykernel_2951/3686781234.py:25: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rh['variable'] = 'rh'
/tmp/ipykernel_2951/3686781234.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_vpd['variable'] = 'vpd'
/tmp/ipykernel_2951/3686781234.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rh.rename(columns=

In [ ]:
from multiprocessing import Pool
import time
max_yrmo = 0

def process_batch(batch_pids):
    """Process a batch of points in parallel."""
    conn2t_local = sqlite3.connect(sqlite_path_2t)
    conn2d_local = sqlite3.connect(sqlite_path_2d)
    
    try:
        pid_list = ','.join(map(str, batch_pids))
        
        # Load batch data
        t_batch = pd.read_sql_query(
            f"SELECT * FROM {table_name} WHERE point_id IN ({pid_list}) AND datetime > {max_yrmo}",
            conn2t_local
        )
        td_batch = pd.read_sql_query(
            f"SELECT * FROM {table_name} WHERE point_id IN ({pid_list}) AND datetime > {max_yrmo}",
            conn2d_local
        )
        
        # Merge and compute
        df_batch = pd.merge(t_batch, td_batch, on='datetime', suffixes=('_t', '_td'))
        cols = ['0', '3', '6', '9', '12', '15', '18', '21']
        
        for col in cols:
            df_batch[f"rh_{col}"] = rh_from_t_td(df_batch[f"{col}_t"], df_batch[f"{col}_td"], units="K")
            df_batch[f"vpd_{col}"] = vpd_from_t_td(df_batch[f"{col}_t"], df_batch[f"{col}_td"], units="K", out_units="kPa")
        
        # Reshape for RH and VPD tables
        df_rh = df_batch[['datetime', 'point_id_t', 'rh_0', 'rh_3', 'rh_6', 'rh_9', 'rh_12', 'rh_15', 'rh_18', 'rh_21']]
        df_rh['variable'] = 'rh'
        df_rh.rename(columns={'point_id_t': 'point_id'}, inplace=True)
        for col in cols:
            df_rh.rename(columns={f"rh_{col}": f"{col}"}, inplace=True)
        
        df_vpd = df_batch[['datetime', 'point_id_t', 'vpd_0', 'vpd_3', 'vpd_6', 'vpd_9', 'vpd_12', 'vpd_15', 'vpd_18', 'vpd_21']]
        df_vpd['variable'] = 'vpd'
        df_vpd.rename(columns={'point_id_t': 'point_id'}, inplace=True)
        for col in cols:
            df_vpd.rename(columns={f"vpd_{col}": f"{col}"}, inplace=True)
        
        return df_rh, df_vpd, len(batch_pids)
    finally:
        conn2t_local.close()
        conn2d_local.close()


# Split points into batches
npoints = len(latlons)
batch_size = 100
batch_list = []
for i in range(0, npoints, batch_size):
    batch_end = min(i + batch_size, npoints)
    batch_pids = latlons.iloc[i:batch_end]['point'].tolist()
    batch_list.append(batch_pids)

print(f"Processing {npoints} points in {len(batch_list)} batches of ~{batch_size} points")
print(f"Starting parallel processing with 8 workers...")

start_time = time.time()
with Pool(8) as pool:
    results = pool.map(process_batch, batch_list)

# Collect results
all_rh = []
all_vpd = []
total_processed = 0
for df_rh, df_vpd, batch_count in results:
    all_rh.append(df_rh)
    all_vpd.append(df_vpd)
    total_processed += batch_count
    print(f"Processed batch: {batch_count} points")

# Concatenate all results
df_all_rh = pd.concat(all_rh, ignore_index=True)
df_all_vpd = pd.concat(all_vpd, ignore_index=True)

print(f"\nTotal rows RH: {len(df_all_rh)}, VPD: {len(df_all_vpd)}")

# Bulk write to SQLite
print("Writing to SQLite...")
with sqlite3.connect(sqlite_path_out_rh) as conn:
    df_all_rh.to_sql(table_name, conn, index=False, if_exists='append')
with sqlite3.connect(sqlite_path_out_vpd) as conn:
    df_all_vpd.to_sql(table_name, conn, index=False, if_exists='append')

elapsed = time.time() - start_time
print(f"\n✓ Completed in {elapsed:.1f} seconds ({elapsed/60:.1f} minutes)")

Processing 2982 points in 30 batches of ~100 points
Starting parallel processing with 8 workers...


In [10]:
td.head()

,datetime,point_id,variable,0,3,6,9,12,15,18,21
0,19900101,0,Td,262.353409,259.617676,257.903488,256.638229,257.118195,255.789139,261.044235,264.886627
1,19900102,0,Td,263.431534,261.733475,260.900864,259.767883,258.295105,258.856216,262.467926,264.387329
2,19900103,0,Td,265.238525,261.455109,258.668167,257.396942,256.466125,255.670944,255.733994,257.049973
3,19900104,0,Td,255.817886,253.556915,251.635101,251.010773,252.482224,252.275360,256.704620,260.231705
4,19900105,0,Td,259.457764,257.022171,258.610458,259.885254,260.098099,260.335129,261.910919,262.417160


In [13]:
df.columns

Index(['datetime', 'point_id_t', 'variable_t', '0_t', '3_t', '6_t', '9_t',
       '12_t', '15_t', '18_t', '21_t', 'point_id_td', 'variable_td', '0_td',
       '3_td', '6_td', '9_td', '12_td', '15_td', '18_td', '21_td'],
      dtype='object')